# Model Report: Hybrid Report + Tutorial

This notebook summarizes the final California AQI forecasting experiment while also explaining the important technical decisions behind the results. It is written as a compact academic report with tutorial-style notes for readers who want to understand how the evaluation was built.

**Research objective.** Predict hourly AQI for California locations under two settings: short-term autoregressive nowcasting and 24-hour-ahead forecasting.

**Reader outcome.** By the end, you should understand what data artifacts are used, how the split avoids leakage, how to read the benchmark metrics, and why the 24-hour task is much harder than the near-term task.


## 1. Setup and Artifact Availability Check

The notebook does not retrain models. Instead, it reads the reproducible CSV outputs produced by the training pipeline and the figure files prepared for the paper/report.

**Learning notes.** Keeping model training outside the report notebook makes the report faster to rerun and easier to audit. A notebook like this should explain and verify artifacts, not hide expensive training steps inside presentation code.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebook':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'processed'
PLOTS = ROOT / 'data' / 'plots'

REQUIRED_DATA_FILES = {
    'Model-ready panel': DATA / 'california_aqi_model_ready.csv',
    'Global leaderboard': DATA / 'california_aqi_model_leaderboard.csv',
    'Scenario evaluation': DATA / 'california_aqi_scenario_evaluation.csv',
    'Leakage audit': DATA / 'california_aqi_leakage_audit.csv',
}

EXPECTED_PLOT_FILES = {
    'Station/source map': PLOTS / 'paper_station_map.png',
    'Correlation heatmap': PLOTS / 'paper_correlation_heatmap_en.png',
    'Workflow pipeline': PLOTS / 'paper_workflow_pipeline_architecture.png',
    'Mutual information features': PLOTS / 'paper_mutual_information_features.png',
    'January 2025 stress window': PLOTS / 'paper_january_2025_stress_window.png',
    'LightGBM predicted vs actual': PLOTS / 'paper_lightgbm_predicted_vs_actual.png',
    'Model leaderboard R2': PLOTS / 'paper_model_leaderboard_r2.png',
}


def build_artifact_table(artifacts, required):
    rows = []
    for label, path in artifacts.items():
        exists = path.exists()
        rows.append({
            'artifact': label,
            'required': required,
            'available': exists,
            'path': str(path.relative_to(ROOT)),
            'size_kb': round(path.stat().st_size / 1024, 1) if exists and path.is_file() else np.nan,
        })
    return pd.DataFrame(rows)

artifact_status = pd.concat([
    build_artifact_table(REQUIRED_DATA_FILES, required=True),
    build_artifact_table(EXPECTED_PLOT_FILES, required=False),
], ignore_index=True)

print('Artifact availability check')
display(artifact_status)

missing_required = artifact_status.query('required == True and available == False')
if not missing_required.empty:
    missing = ', '.join(missing_required['path'].tolist())
    raise FileNotFoundError(f'Missing required data artifact(s): {missing}')


## 2. Dataset Overview

The model-ready file is an hourly panel. Each row represents one station/source at one timestamp, with pollutant measurements, meteorological variables, calendar fields, and the target AQI value.

**Learning notes.** A panel dataset combines time-series behavior with cross-location behavior. That matters here because AQI has both local temporal persistence and regional spatial patterns.


In [ ]:
model_ready = pd.read_csv(REQUIRED_DATA_FILES['Model-ready panel'], parse_dates=['time'])
leaderboard = pd.read_csv(REQUIRED_DATA_FILES['Global leaderboard'])
scenario = pd.read_csv(REQUIRED_DATA_FILES['Scenario evaluation'])
audit = pd.read_csv(REQUIRED_DATA_FILES['Leakage audit'])

summary = pd.DataFrame({
    'metric': [
        'rows',
        'columns',
        'stations',
        'sources',
        'start_time',
        'end_time',
        'total_missing_values',
    ],
    'value': [
        f'{len(model_ready):,}',
        f'{model_ready.shape[1]:,}',
        f'{model_ready["station_id"].nunique():,}',
        ', '.join(sorted(model_ready['source'].dropna().unique())),
        model_ready['time'].min(),
        model_ready['time'].max(),
        f'{int(model_ready.isna().sum().sum()):,}',
    ],
})

display(summary)

station_coverage = (
    model_ready.groupby(['source', 'station_id'])
    .agg(
        first_time=('time', 'min'),
        last_time=('time', 'max'),
        rows=('time', 'size'),
        mean_target_aqi=('target_aqi', 'mean'),
        p95_target_aqi=('target_aqi', lambda s: s.quantile(0.95)),
    )
    .reset_index()
    .sort_values(['source', 'station_id'])
)

display(station_coverage)


## 3. Forecasting Tasks and Evaluation Protocol

The experiment compares two practical prediction settings:

1. **Short-term autoregressive nowcasting (Lag 1-3h):** predict near-future AQI using very recent local AQI history.
2. **Long-term forecasting (Lag 24h):** predict AQI one day ahead, where recent local target information is restricted.

**Learning notes.** These tasks should not be interpreted as equally difficult. The nowcasting task benefits from short-term autocorrelation: if AQI was high one hour ago, it is often still high now. The 24-hour task is closer to a planning forecast, so it depends more on weather, seasonal context, and broader spatial signals.


In [ ]:
protocol_checks = [
    'split_strategy',
    'train_years',
    'validation_years',
    'test_years',
    'post_validation_guard_hours',
    'train_target_period',
    'validation_target_period',
    'test_target_period',
    'test_station_distribution',
]

protocol = (
    audit[audit['check'].isin(protocol_checks)]
    .loc[:, ['configuration', 'check', 'value', 'interpretation']]
    .sort_values(['configuration', 'check'])
)

display(protocol)


## 4. Feature Engineering and Leakage Controls

The final pipeline combines three groups of predictors:

- **Meteorological context:** temperature, humidity, rain, cloud cover, wind, and pressure.
- **Temporal context:** hour, day of week, month, and year.
- **Lagged AQI context:** local and spatial lag features, constrained differently for nowcasting and 24-hour forecasting.

The leakage audit is important because lag features can easily make a forecasting task unrealistically easy if future or same-time target information slips into the feature matrix.

**Learning notes.** Leakage is not only a coding bug. It can also be a modeling-design mistake. If a 24-hour forecast uses target information from one hour ago, the reported score may look impressive while failing to represent the real deployment scenario.


In [ ]:
lag_checks = [
    'future_like_feature_names',
    'current_pollutant_proxy_features_present',
    'minimum_lag_hours',
    'minimum_local_target_lag_hours',
    'minimum_spatial_lag_hours',
    'recent_local_target_lag_features_under_24h',
    'intermediate_spatial_lag_features_6_12h',
    'corr_y_test_with_target_aqi_lag_1',
    'corr_y_test_with_target_aqi_lag_24',
]

lag_audit = (
    audit[audit['check'].isin(lag_checks)]
    .loc[:, ['configuration', 'check', 'value', 'interpretation']]
    .sort_values(['configuration', 'check'])
)

display(lag_audit)


## 5. Global Model Leaderboard

The leaderboard compares Linear Ridge, Random Forest, LightGBM, XGBoost, and CatBoost under both forecasting configurations.

### How to read MAE, RMSE, and R2

- **MAE** is the average absolute error in AQI points. Lower is better and easier to interpret directly.
- **RMSE** penalizes large errors more strongly than MAE. Lower is better, especially when extreme AQI mistakes matter.
- **R2** measures explained variance relative to a baseline. Higher is better; values closer to 1 indicate stronger predictive fit.

**Learning notes.** Use all three metrics together. A model with strong R2 but high RMSE may track normal conditions well while still missing extreme episodes.


In [ ]:
metric_columns = ['MAE', 'RMSE', 'R2_Score', 'Runtime_sec']
leaderboard_sorted = leaderboard.sort_values(['Configuration', 'R2_Score'], ascending=[True, False]).copy()

display(
    leaderboard_sorted.assign(**{col: leaderboard_sorted[col].round(4) for col in metric_columns})
)

best_by_configuration = (
    leaderboard_sorted.groupby('Configuration', as_index=False)
    .first()
    .loc[:, ['Configuration', 'Model', 'N', 'MAE', 'RMSE', 'R2_Score', 'Runtime_sec']]
)

print('Best model per configuration by R2 score')
display(best_by_configuration.assign(
    MAE=best_by_configuration['MAE'].round(4),
    RMSE=best_by_configuration['RMSE'].round(4),
    R2_Score=best_by_configuration['R2_Score'].round(4),
    Runtime_sec=best_by_configuration['Runtime_sec'].round(2),
))


In [ ]:
configs = leaderboard_sorted['Configuration'].drop_duplicates().tolist()
fig, axes = plt.subplots(1, len(configs), figsize=(6 * len(configs), 4), sharey=True)
if len(configs) == 1:
    axes = [axes]

for ax, cfg in zip(axes, configs):
    sub = leaderboard_sorted[leaderboard_sorted['Configuration'] == cfg].sort_values('R2_Score')
    colors = ['#6b7280'] * len(sub)
    colors[-1] = '#2563eb'
    ax.barh(sub['Model'], sub['R2_Score'], color=colors)
    ax.set_title(cfg)
    ax.set_xlabel('R2 score')
    ax.set_xlim(0, max(1.0, leaderboard_sorted['R2_Score'].max() + 0.05))
    for y, value in enumerate(sub['R2_Score']):
        ax.text(value + 0.01, y, f'{value:.3f}', va='center', fontsize=9)

fig.suptitle('Global benchmark: R2 by model and forecasting task', y=1.04, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(configs), figsize=(6 * len(configs), 4), sharey=False)
if len(configs) == 1:
    axes = [axes]

for ax, cfg in zip(axes, configs):
    sub = leaderboard_sorted[leaderboard_sorted['Configuration'] == cfg].sort_values('MAE')
    ax.bar(sub['Model'], sub['MAE'], color='#059669')
    ax.set_title(cfg)
    ax.set_ylabel('MAE (AQI points)')
    ax.tick_params(axis='x', rotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')

fig.suptitle('Lower MAE means smaller average AQI error', y=1.04, fontsize=14)
plt.tight_layout()
plt.show()


## 6. Scenario-Level Robustness

The scenario evaluation checks whether model behavior changes across operating conditions: the full 2025 test year, non-event AQI, a January stress-test window, top-5 percent AQI observations, and wildfire-season months.

**Learning notes.** Scenario-level robustness is different from average leaderboard performance. A model can perform well overall but still be weak during rare high-AQI events, which are often the cases that matter most for public-health decision support.


In [ ]:
available_scenarios = scenario[scenario['Scenario_Status'].eq('Available')].copy()
available_scenarios = available_scenarios.sort_values(['Configuration', 'Scenario', 'R2_Score'], ascending=[True, True, False])

display(available_scenarios.assign(
    MAE=available_scenarios['MAE'].round(4),
    RMSE=available_scenarios['RMSE'].round(4),
    R2_Score=available_scenarios['R2_Score'].round(4),
))

scenario_best = available_scenarios.loc[
    available_scenarios.groupby(['Configuration', 'Scenario'])['R2_Score'].idxmax()
].sort_values(['Configuration', 'Scenario'])

print('Best model within each scenario by R2 score')
display(scenario_best.loc[:, ['Configuration', 'Scenario', 'Model', 'N', 'MAE', 'RMSE', 'R2_Score']].assign(
    MAE=scenario_best['MAE'].round(4),
    RMSE=scenario_best['RMSE'].round(4),
    R2_Score=scenario_best['R2_Score'].round(4),
))


In [ ]:
for cfg in configs:
    sub = scenario_best[scenario_best['Configuration'] == cfg].sort_values('R2_Score')
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.barh(sub['Scenario'], sub['R2_Score'], color='#7c3aed')
    ax.set_title(f'Scenario robustness: {cfg}')
    ax.set_xlabel('Best available R2 score')
    ax.axvline(0, color='black', linewidth=0.8)
    for y, value in enumerate(sub['R2_Score']):
        ax.text(value + 0.02 if value >= 0 else value - 0.08, y, f'{value:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()


## 7. Paper Figures

The report can display any prepared paper figures that are present in `data/plots`. Missing optional figures are listed explicitly so the reader knows whether an expected visualization was unavailable or simply omitted.


In [ ]:
figure_catalog = pd.DataFrame([
    {
        'figure': label,
        'filename': path.name,
        'available': path.exists(),
        'path': str(path.relative_to(ROOT)),
    }
    for label, path in EXPECTED_PLOT_FILES.items()
])

display(figure_catalog)

for label, path in EXPECTED_PLOT_FILES.items():
    if path.exists():
        print(f'{label}: {path.relative_to(ROOT)}')
        display(Image(filename=str(path)))
    else:
        print(f'Missing optional figure: {path.relative_to(ROOT)}')


## 8. Leakage audit interpretation

The leakage audit translates modeling assumptions into explicit checks. The most important checks confirm the year split, the validation-to-train guard window, the absence of future-like feature names, and the minimum lag constraints for each forecasting task.

**Learning notes.** A strong benchmark is not only about getting a high score. It is about proving that the score came from information that would have been available at prediction time.


In [ ]:
key_audit_checks = [
    'target_aqi_is_pm25_proxy',
    'split_strategy',
    'post_validation_guard_hours',
    'future_like_feature_names',
    'current_pollutant_proxy_features_present',
    'minimum_local_target_lag_hours',
    'minimum_spatial_lag_hours',
    'ridge_all_features_r2',
    'ridge_without_pm_or_target_history_r2',
]

key_audit = (
    audit[audit['check'].isin(key_audit_checks)]
    .loc[:, ['configuration', 'check', 'value', 'interpretation']]
    .sort_values(['configuration', 'check'])
)

display(key_audit)


In [ ]:
def audit_value(configuration, check):
    rows = audit[(audit['configuration'].eq(configuration)) & (audit['check'].eq(check))]
    if rows.empty:
        return None
    return rows.iloc[0]['value']

for cfg in configs:
    all_features_r2 = audit_value(cfg, 'ridge_all_features_r2')
    reduced_features_r2 = audit_value(cfg, 'ridge_without_pm_or_target_history_r2')
    local_lag = audit_value(cfg, 'minimum_local_target_lag_hours')
    spatial_lag = audit_value(cfg, 'minimum_spatial_lag_hours')
    print(cfg)
    print(f'  Minimum local target lag: {local_lag} hour(s)')
    print(f'  Minimum spatial lag: {spatial_lag} hour(s)')
    print(f'  Ridge R2 with all safe features: {float(all_features_r2):.3f}')
    print(f'  Ridge R2 without PM/target history: {float(reduced_features_r2):.3f}')
    print()


## 9. Academic Interpretation and Takeaways

1. **Short-term nowcasting is the easier task.** The best short-term model reaches much higher R2 because recent AQI lags carry strong information about the near future.

2. **The 24-hour forecast is more realistic for planning but harder.** Removing short local target history lowers the amount of directly predictive information available to the model.

3. **Extreme AQI remains the key weakness.** Scenario-level metrics show that high-AQI slices can be much harder than ordinary conditions, even when full-year scores look strong.

4. **Leakage controls make the comparison credible.** The split, guard windows, and lag constraints are essential because time-series air-quality data can otherwise produce inflated scores.

5. **The strongest next step is event-focused modeling.** Future work should emphasize wildfire-season and high-AQI regimes, potentially using event labels, additional atmospheric transport features, or specialized loss functions for extremes.
